# 03 – Model Training & Evaluation (Master Notebook)

Notebook này là điểm chạy chính trên Google Colab.

Khi chọn **Runtime → Run all**, notebook sẽ:
1. Chuẩn bị repository và thư viện.
2. Thực thi `01_EDA.ipynb`.
3. Thực thi `02_Data_Preprocessing.ipynb`.
4. Gọi các model có sẵn trong `modules/model_training.py`: Logistic Regression, SVM và Random Forest.
5. Lưu model tốt nhất và các báo cáo đánh giá.

Hai notebook đầu vẫn được giữ riêng để nhóm có thể mở và chạy độc lập. Notebook này dùng `nbclient` để thực thi chúng theo đúng thứ tự trong cùng project.


## 0. Environment setup

Nếu notebook được mở trực tiếp từ GitHub, cell dưới sẽ clone repository vào Colab. Nếu project đã có trong runtime, nó sẽ tìm và sử dụng thư mục hiện tại.


In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DuyTuong101/credit-risk-prediction.git"
REPO_DIR_NAME = "credit-risk-prediction"

def find_project_root(start):
    for candidate in [start, *start.parents]:
        if (
            (candidate / "modules" / "data_processing.py").is_file()
            and (candidate / "modules" / "model_training.py").is_file()
            and (candidate / "notebooks").is_dir()
            and (candidate / "data").is_dir()
        ):
            return candidate
    return None

PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None:
    clone_dir = Path("/content") / REPO_DIR_NAME
    if not clone_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    PROJECT_ROOT = find_project_root(clone_dir)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate project root. Verify REPO_URL and repository access."
    )

NOTEBOOK_DIR = PROJECT_ROOT / "notebooks"
FEATURES_DIR = PROJECT_ROOT / "features"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

for folder in [FEATURES_DIR, PROCESSED_DIR, MODELS_DIR, REPORTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version)


## 1. Install dependencies

Cài thư viện từ `requirements.txt` của repository. `nbclient` được dùng để chạy tuần tự hai notebook đầu.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt"), "nbclient"],
    check=True
)

import nbformat
from nbclient import NotebookClient

print("Dependencies are ready.")


## 2. Run Notebook 1 – EDA

Notebook 1 được thực thi trước. Nếu có cell lỗi, quá trình sẽ dừng tại đó để tránh chạy tiếp trên dữ liệu chưa được kiểm tra.


In [ ]:
def execute_notebook(notebook_path: Path, timeout_seconds: int = 3600):
    print(f"\n{'=' * 75}")
    print("Executing:", notebook_path.name)
    print(f"{'=' * 75}")

    notebook = nbformat.read(str(notebook_path), as_version=4)
    client = NotebookClient(
        notebook,
        timeout=timeout_seconds,
        kernel_name="python3",
        allow_errors=False,
        resources={"metadata": {"path": str(NOTEBOOK_DIR)}},
    )
    executed = client.execute()
    print("Completed:", notebook_path.name)
    return executed

eda_path = NOTEBOOK_DIR / "01_EDA.ipynb"
if not eda_path.is_file():
    raise FileNotFoundError(f"Missing notebook: {eda_path}")

eda_executed = execute_notebook(eda_path)


## 3. Run Notebook 2 – Data Preprocessing

Notebook 2 làm sạch dữ liệu, chia train/test và tạo các cấu hình feature `.npy`. Các file được lưu trong thư mục `features/`.


In [ ]:
preprocessing_path = NOTEBOOK_DIR / "02_Data_Preprocessing.ipynb"
if not preprocessing_path.is_file():
    raise FileNotFoundError(f"Missing notebook: {preprocessing_path}")

preprocessing_executed = execute_notebook(preprocessing_path)


## 4. Train the existing models from the repository

Không định nghĩa lại model ở notebook này. Ta import `train_and_evaluate()` từ `modules/model_training.py`, module hiện có của nhóm.

Module tự thực hiện preprocessing trong pipeline của nó, sau đó so sánh Logistic Regression, SVM và Random Forest bằng Stratified 3-fold Cross Validation. Model có CV F1 trung bình cao nhất sẽ được fit và đánh giá trên held-out test set.


In [ ]:
import json
import pandas as pd
from IPython.display import display
from modules.model_training import train_and_evaluate

CLEANED_DATA_PATH = PROCESSED_DIR / "credit_risk_cleaned.csv"
if not CLEANED_DATA_PATH.is_file():
    raise FileNotFoundError(
        "The cleaned dataset was not produced by 02_Data_Preprocessing.ipynb."
    )

df_clean = pd.read_csv(CLEANED_DATA_PATH)
print("Cleaned dataset shape:", df_clean.shape)
print("Target distribution:")
display(df_clean["loan_status"].value_counts(normalize=True).rename("proportion"))

training_results = train_and_evaluate(
    df=df_clean,
    target_col="loan_status",
    test_size=0.2,
    cv_folds=3,
    random_state=42,
    scaler="standard",
    numeric_imputation="median",
    categorical_imputation="most_frequent",
    model_names=(
        "logistic_regression",
        "svm",
        "random_forest",
    ),
    models_dir=MODELS_DIR,
    reports_dir=REPORTS_DIR,
)

print("Cross-validation comparison (sorted by mean F1):")
display(training_results["comparison"])

print("Held-out test metrics:")
display(pd.DataFrame([training_results["test_metrics"]]).T.rename(columns={0: "value"}))
print("Best model saved to:", training_results["model_path"])


## 5. Verify generated deliverables

Kiểm tra model, metric reports và feature files sau khi chạy hai notebook đầu và bước training.


In [ ]:
expected_outputs = [
    MODELS_DIR / "best_model.joblib",
    REPORTS_DIR / "model_comparison.csv",
    REPORTS_DIR / "test_metrics.json",
    REPORTS_DIR / "classification_report.json",
    REPORTS_DIR / "confusion_matrix.csv",
    FEATURES_DIR / "X_train_baseline.npy",
    FEATURES_DIR / "X_test_baseline.npy",
    FEATURES_DIR / "y_train.npy",
    FEATURES_DIR / "y_test.npy",
]

verification = pd.DataFrame({
    "file": [str(p.relative_to(PROJECT_ROOT)) for p in expected_outputs],
    "exists": [p.is_file() for p in expected_outputs],
})
display(verification)

missing = verification.loc[~verification["exists"], "file"].tolist()
if missing:
    raise FileNotFoundError("Missing expected output(s): " + ", ".join(missing))

print("\nAll required outputs in this notebook's verification list exist.")


## 6. Important note about preprocessing experiments

`02_Data_Preprocessing.ipynb` tạo 8 bộ feature arrays `.npy`. Module `modules/model_training.py` hiện tại lại nhận DataFrame và tự xây dựng preprocessing pipeline riêng bên trong cross-validation. Vì vậy, các `.npy` hiện được tạo và lưu làm deliverables nhưng chưa được đưa trực tiếp vào model training.

Notebook này giữ nguyên thiết kế model hiện có của nhóm, không tự ý thay đổi module. Ở bước hoàn thiện thí nghiệm, nhóm nên quyết định có mở rộng training để so sánh các preprocessing configurations hay giữ kết quả model training hiện tại như một thí nghiệm riêng.


In [ ]:
print("Pipeline completed.")
print("EDA notebook:", eda_path)
print("Preprocessing notebook:", preprocessing_path)
print("Model comparison:", REPORTS_DIR / "model_comparison.csv")
print("Test metrics:", REPORTS_DIR / "test_metrics.json")
print("Best model:", MODELS_DIR / "best_model.joblib")
